In [1]:
import pandas as pd
import urllib.request
import zipfile
import io
import csv

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

with urllib.request.urlopen(url, timeout=60) as response:
    zip_data = response.read()

with zipfile.ZipFile(io.BytesIO(zip_data)) as archive:
    with archive.open("SMSSpamCollection") as file:
        df = pd.read_csv(
            file,
            sep="\t",
            header=None,
            names=["label", "message"],
            quoting=csv.QUOTE_NONE
        )

print("Number of messages:", len(df))
display(df.head())

Number of messages: 5574


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [2]:
print("Messages per category:")
print(df["label"].value_counts())

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate messages:")
print(df.duplicated(subset=["message"]).sum())

Messages per category:
label
ham     4827
spam     747
Name: count, dtype: int64

Missing values:
label      0
message    0
dtype: int64

Duplicate messages:
403


In [3]:
# Remove missing values and repeated messages
clean_df = df.dropna(subset=["label", "message"]).copy()

clean_df = clean_df.drop_duplicates(
    subset=["message"]
).reset_index(drop=True)

print("Before cleaning:", len(df))
print("After cleaning:", len(clean_df))

print("\nMessages per category:")
print(clean_df["label"].value_counts())

print("\nRemaining duplicates:")
print(clean_df.duplicated(subset=["message"]).sum())

Before cleaning: 5574
After cleaning: 5171

Messages per category:
label
ham     4518
spam     653
Name: count, dtype: int64

Remaining duplicates:
0


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

X = clean_df["message"]
y = clean_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", MultinomialNB())
])

model.fit(X_train, y_train)

print("Training messages:", len(X_train))
print("Testing messages:", len(X_test))
print("Model trained successfully!")
from sklearn.metrics import accuracy_score, classification_report

predictions = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, predictions):.2%}")

print("\nClassification report:")
print(classification_report(
    y_test,
    predictions,
    digits=3,
    zero_division=0
))

Training messages: 4136
Testing messages: 1035
Model trained successfully!
Accuracy: 95.27%

Classification report:
              precision    recall  f1-score   support

         ham      0.949     1.000     0.974       904
        spam      1.000     0.626     0.770       131

    accuracy                          0.953      1035
   macro avg      0.974     0.813     0.872      1035
weighted avg      0.955     0.953     0.948      1035



In [7]:
message = input("Enter a message in English: ").strip()

if not message:
    print("Please enter a message.")
else:
    prediction = model.predict([message])[0]

    if prediction == "spam":
        print("Prediction: SPAM — unwanted message")
    else:
        print("Prediction: HAM — normal message")

Enter a message in English: Congratulations! You won a free prize. Call now to claim!
Prediction: SPAM — unwanted message


Project Results and Limitations

Dataset:
UCI SMS Spam Collection.
After removing duplicate messages, 5,171 messages remained.
Training set: 4,136 messages.
Test set: 1,035 messages.

Model:
TF-IDF with Multinomial Naive Bayes.

Test Results:
Accuracy: 95.27%
Spam precision: 100%
Spam recall: 62.6%
Spam F1-score: 0.770

The model detected 82 of 131 spam messages.
It missed 49 spam messages.
No normal messages were classified as spam in this test.

Manual Examples:
"hello mohamad" was classified as HAM.
"Congratulations! You won a free prize. Call now to claim!"
was classified as SPAM.

Limitations:
The model can miss spam messages.
These results apply to this test split, not all future messages.
The project is intended for English SMS classification.
A HAM prediction does not guarantee that a message is safe.

Dataset source:
https://doi.org/10.24432/C5CC84